# 009-guided-filter-refinement

# Imports

In [ ]:
import itertools
from dataclasses import asdict

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from uavloc.data.annotated import IGNORE_LABEL
from uavloc.features.segmentation import CLASSES
from uavloc.plotting.correlation import SEG_PALETTE
from uavloc.segmentation.boundary import label_band
from uavloc.segmentation.eval import (MEAN_KEY, VAL_SETS, bilinear_argmax, evaluate, load_baseline_model,
                                      load_val_sets, lraspp_outputs, results_table, run_model)
from uavloc.segmentation.latency import refiner_latency, standard_tile
from uavloc.segmentation.refine.guided_filter import GuidedFilterConfig, refiner
from uavloc.track import experiment_dir, log_run, set_seed

# Params

In [ ]:
EXP = "009-guided-filter-refinement"
NOTE = ""
LOG_RUN = True

MODE = "baseline"               # "baseline" | "full_res" | "fast"
GUIDE = "rgb"                   # "rgb" | "grey"
RADII = [2, 4, 8]               # full_res: px; fast: stride-8 cells
EPS_GRID = [1e-3, 1e-2, 1e-1]   # guide in [0, 1], so intensity^2 units

SELECT_SET = "birky-strip"
MIOU_GUARD = 0.01
BOUNDARY_PX = 4
BF_TOLERANCES_PX = [1, 2]
SEED = 42
DEVICE = "cpu"

LATENCY_SET = "bushcha"
LATENCY_TOP_LEFT_PX = [1024, 1024]
LATENCY_THREADS = 4
LATENCY_WARMUP = 5
LATENCY_RUNS = 30

ZOOMS = [["birky-strip-google", 0, [120, 1400, 376, 1912]], ["bushcha", 0, [1024, 1024, 1280, 1536]]]
HEATMAP_DELTA_RANGE = 0.02

# Data

In [ ]:
set_seed(SEED)
TOLERANCES = tuple(BF_TOLERANCES_PX)
RUN_CONFIG = {"mode": MODE, "guide": GUIDE if MODE != "baseline" else None, "radii": RADII, "eps_grid": EPS_GRID,
              "select_set": SELECT_SET, "miou_guard": MIOU_GUARD, "boundary_px": BOUNDARY_PX,
              "bf_tolerances_px": BF_TOLERANCES_PX, "latency_threads": LATENCY_THREADS}

In [ ]:
val_sets = load_val_sets()
{name: len(items) for name, items in val_sets.items()}

# Model

In [ ]:
model = load_baseline_model(DEVICE)

In [ ]:
cases = run_model(model, val_sets, DEVICE)

# Run

In [ ]:
baseline = evaluate(bilinear_argmax, cases, BOUNDARY_PX, TOLERANCES)
results_table(baseline).round(3)

In [ ]:
def grid_configs() -> list[GuidedFilterConfig]:
    """Every (radius, eps) of the registered grid; none for the baseline run."""
    if MODE == "baseline":
        return []
    return [GuidedFilterConfig(MODE, GUIDE, r, eps) for r, eps in itertools.product(RADII, EPS_GRID)]

In [ ]:
grid = {config: evaluate(refiner(config), cases, BOUNDARY_PX, TOLERANCES) for config in grid_configs()}
len(grid)

In [ ]:
def grid_row(config: GuidedFilterConfig, results: dict, set_name: str) -> dict:
    """One grid point on one set: scalar metrics and per-class Boundary IoU."""
    r = results[set_name]
    per_class = {f"biou_{c}": v for c, v in r.get("biou_per_class", {}).items()}
    return {"radius": config.radius, "eps": config.eps, "set": set_name,
            **{k: v for k, v in r.items() if not isinstance(v, dict)}, **per_class}

In [ ]:
grid_table = pd.DataFrame([grid_row(c, res, s) for c, res in grid.items() for s in [*VAL_SETS, MEAN_KEY]])
grid_table.pivot_table(index=["radius", "eps"], columns="set", values=["miou", "biou"]).round(3) if grid else None

In [ ]:
def select(grid: dict) -> GuidedFilterConfig | None:
    """Max Boundary IoU on SELECT_SET among configs within MIOU_GUARD of the baseline mIoU there."""
    floor = baseline[SELECT_SET]["miou"] - MIOU_GUARD
    allowed = {c: r[SELECT_SET]["biou"] for c, r in grid.items() if r[SELECT_SET]["miou"] >= floor}
    return max(allowed, key=allowed.get) if allowed else None

In [ ]:
selected = select(grid)
chosen = grid[selected] if selected else baseline
chosen_fn = refiner(selected) if selected else bilinear_argmax
selected

In [ ]:
summary = pd.concat({"baseline": results_table(baseline), "selected": results_table(chosen)}, names=["run", "set"])
summary.round(3)

In [ ]:
per_class = pd.concat({run: pd.DataFrame({n: r[n]["biou_per_class"] for n in VAL_SETS}).T
                       for run, r in {"baseline": baseline, "selected": chosen}.items()}, names=["run", "set"])
per_class.round(3)

In [ ]:
latency_tile = standard_tile(val_sets[LATENCY_SET][0].rgb, *LATENCY_TOP_LEFT_PX)
latency_output = lraspp_outputs(model, latency_tile, DEVICE)
timing = dict(warmup=LATENCY_WARMUP, runs=LATENCY_RUNS, threads=LATENCY_THREADS)
argmax_latency = refiner_latency(bilinear_argmax, latency_output, latency_tile, **timing)
latency = refiner_latency(chosen_fn, latency_output, latency_tile, **timing)
latency, argmax_latency

In [ ]:
def run_metrics() -> dict[str, float]:
    """Selected config's scores per set and mean, deltas vs baseline, latency, per-class BIoU."""
    metrics = {f"{k}_{name}": v for name in [*VAL_SETS, MEAN_KEY] for k, v in chosen[name].items()
               if not isinstance(v, dict)}
    metrics.update({f"delta_{k}_{name}": chosen[name][k] - baseline[name][k]
                    for name in [*VAL_SETS, MEAN_KEY] for k in ("miou", "biou")})
    metrics.update({f"biou_{c}_{name}": v for name in VAL_SETS for c, v in chosen[name]["biou_per_class"].items()})
    metrics.update({"refiner_ms_median": latency.median_ms, "refiner_ms_p10": latency.p10_ms,
                    "refiner_ms_p90": latency.p90_ms, "argmax_ms_median": argmax_latency.median_ms,
                    "refiner_to_argmax_ratio": latency.median_ms / argmax_latency.median_ms})
    return metrics

In [ ]:
metrics = run_metrics()
RUN_CONFIG["selected"] = asdict(selected) if selected else None
{k: round(v, 4) for k, v in metrics.items() if k.endswith(MEAN_KEY) or k.startswith("refiner")}

# Plots

In [ ]:
def colour(lab: np.ndarray) -> np.ndarray:
    out = np.full((*lab.shape, 3), 255, np.uint8)
    ok = lab != IGNORE_LABEL
    out[ok] = SEG_PALETTE[lab[ok]]
    return out

In [ ]:
def band_overlay(pred: np.ndarray, label: np.ndarray) -> np.ndarray:
    """Predicted classes, with the label's 1-px class contours drawn in white."""
    out = colour(pred)
    edges = np.any([label_band(label, c, 1) for c in range(len(CLASSES))], axis=0)
    out[edges] = 255
    return out

In [ ]:
def zoom_panels(set_name: str, index: int, box: list[int]) -> list[tuple[np.ndarray, str]]:
    """RGB | label | baseline | selected, cropped to `box` = [top, left, bottom, right]."""
    case = cases[set_name][index]
    zoom = np.s_[box[0]:box[2], box[1]:box[3]]
    lab = case.image.label[zoom]
    return [(case.image.rgb[zoom], f"{set_name} {case.image.date} RGB"), (colour(lab), "label"),
            (band_overlay(bilinear_argmax(case.output, case.image.rgb)[zoom], lab), "baseline (bilinear argmax)"),
            (band_overlay(chosen_fn(case.output, case.image.rgb)[zoom], lab), f"selected: {selected}")]

In [ ]:
fig_zoom, axes = plt.subplots(len(ZOOMS), 4, figsize=(20, 3.2 * len(ZOOMS)), squeeze=False)
for row, (set_name, index, box) in zip(axes, ZOOMS):
    for ax, (img, title) in zip(row, zoom_panels(set_name, index, box)):
        ax.imshow(img, interpolation="nearest"); ax.set_title(title, fontsize=8); ax.axis("off")
fig_zoom.legend([plt.Rectangle((0, 0), 1, 1, color=SEG_PALETTE[c] / 255) for c in range(len(CLASSES))], CLASSES,
                loc="lower center", ncol=len(CLASSES))
fig_zoom.tight_layout(rect=(0, .04, 1, 1))

In [ ]:
fig_grid, axes = plt.subplots(1, len(VAL_SETS), figsize=(4.5 * len(VAL_SETS), 3.5), squeeze=False)
for ax, set_name in zip(axes[0], VAL_SETS):
    if grid:
        heat = grid_table[grid_table["set"] == set_name].pivot(index="radius", columns="eps", values="biou")
        shown = ax.imshow(heat - baseline[set_name]["biou"], cmap="RdBu", vmin=-HEATMAP_DELTA_RANGE, vmax=HEATMAP_DELTA_RANGE)
        ax.set_xticks(range(len(heat.columns)), heat.columns); ax.set_yticks(range(len(heat.index)), heat.index)
        ax.set_xlabel("eps"); ax.set_ylabel("radius")
        fig_grid.colorbar(shown, ax=ax, label="BIoU - baseline")
    ax.set_title(set_name)
fig_grid.tight_layout()

# Log

In [ ]:
run = log_run(exp=EXP, config=RUN_CONFIG, metrics=metrics, note=NOTE, figures=[fig_zoom, fig_grid], seed=SEED,
              device=DEVICE) if LOG_RUN else {"run": "dry"}
artifacts = experiment_dir(EXP) / "artifacts"
artifacts.mkdir(exist_ok=True)
grid_table.to_csv(artifacts / f"grid-{run['run']}.csv", index=False)
run["run"]